# MCP Banking Assistant

An AI assistant that answers banking questions by calling tools on a **Model Context Protocol (MCP) server**.

- The **server** (`server/bank_mcp_server.py`) exposes four read-only tools over the data. Customer IDs are masked before any tool can return them.
- The **assistant** (`assistant/assistant.py`) is an MCP client. With an `ANTHROPIC_API_KEY` it lets Claude choose the tools. Without a key it uses a rule-based router, so the project runs offline.
- A **guardrail** rejects any answer that contains a number no tool returned.

The data is **synthetic**. See `README.md` for the problems this project solves and how.

Run all cells from the project folder. Nothing needs an API key.

## 0. Setup
Check the files and choose the mode.

In [1]:
import os  # environment variables and paths
import sys  # to import the assistant module
import re  # pattern checks for the privacy test
import pandas as pd  # tables for readable output

PROJECT = os.path.abspath('.')  # the folder the notebook runs from
SERVER = os.path.join(PROJECT, 'server', 'bank_mcp_server.py')  # the MCP server file
DATA = os.path.join(PROJECT, 'data', 'bank_transactions.csv')  # the synthetic transaction file
sys.path.insert(0, os.path.join(PROJECT, 'assistant'))  # make assistant.py importable
import assistant as bank_assistant  # the assistant module (the MCP client)

print('Server file found:', os.path.exists(SERVER))  # confirm the server is where we expect
print('Data file found:', os.path.exists(DATA))  # confirm the data is where we expect
print('ANTHROPIC_API_KEY set:', bool(os.environ.get('ANTHROPIC_API_KEY')))  # decides claude or offline mode

Server file found: True
Data file found: True
ANTHROPIC_API_KEY set: False


## 1. Start the MCP server and list its tools
The client starts the server as a subprocess and talks to it over stdio, the standard MCP transport.

In [2]:
assistant = await bank_assistant.BankAssistant(SERVER, DATA).start()  # launch the server and connect
print('Mode:', assistant.mode)  # claude or offline
for t in assistant.tools:  # every tool the server offers
    props = list(t.inputSchema.get('properties', {}).keys())  # its input parameters
    print(f'- {t.name}({", ".join(props)})')  # one line per tool
    print('    ', (t.description or '').strip().split(chr(10))[0])  # first line of its description

Mode: offline
- list_accounts()
     List every account in the data with its currency, number of transactions and masked customer token.
- account_summary(account_id)
     Summarise one account: credits, debits, net flow, success and failure counts, largest debit and date range.
- search_transactions(account_id, status, transaction_type, min_amount, limit)
     Search transactions. All filters are optional: account_id (e.g. ACC2001), status (SUCCESS or FAILED),
- flag_high_value_debits(threshold_mur)
     List successful debits at or above a threshold in MUR (USD converted at a fixed demo rate).


## 2. Call the tools directly
The tools return plain data. The assistant builds on this, so each result can be checked on its own.

In [3]:
summary = await assistant.call('account_summary', {'account_id': 'ACC2003'})  # totals for one account
print(pd.Series(summary).to_string())  # show every field

account_id          ACC2003
currency                MUR
transactions             28
successful               27
failed                    1
credits            73587.31
debits             52018.72
net_flow           21568.59
largest_debit        9990.0
first_date       2026-07-14
last_date        2026-09-29


In [4]:
large = await assistant.call('flag_high_value_debits', {'threshold_mur': 10000})  # large successful debits
pd.DataFrame(large['rows'])[['transaction_id', 'account_id', 'date', 'time', 'amount', 'currency', 'customer_token']]  # show the table

,transaction_id,account_id,date,time,amount,currency,customer_token
0,TXN0209,ACC2001,2026-09-26,09:40,22008.33,MUR,cust_09f61913d4
1,TXN0090,ACC2002,2026-08-14,11:02,12500.00,MUR,cust_6de94e44ff
2,TXN0150,ACC2004,2026-09-03,15:40,10800.00,MUR,cust_ed0c7a1158


In [5]:
loaded = await assistant.call('list_accounts', {})  # accounts plus the data quality report
print('Data quality:', loaded['data_quality'])  # how many rows were set aside or de-duplicated
pd.DataFrame(loaded['accounts'])  # one row per account

Data quality: {'raw_rows': 223, 'set_aside': 3, 'exact_copies_removed': 0, 'usable_rows': 220}


,account_id,currency,transactions,customer_token
0,ACC2001,MUR,35,cust_09f61913d4
1,ACC2002,MUR,44,cust_6de94e44ff
2,ACC2003,MUR,28,cust_bac0dd1336
3,ACC2004,MUR,38,cust_ed0c7a1158
4,ACC2005,MUR,34,cust_7e156a97d6
5,ACC2006,USD,41,cust_21980e5547


## 3. Ask questions
Each question goes through the assistant. In offline mode the router picks the tool. In Claude mode the model picks it.

In [6]:
questions = [  # a mix of totals, lists, searches and an unknown account
    'Give me the summary for ACC2003',
    'Show large debits over 10000',
    'Failed transactions on ACC2002',
    'What accounts do you have?',
    'Summary for ACC9999',
]
for q in questions:  # ask each one in turn
    r = await assistant.ask(q)  # get the answer and the tools used
    print('Q:', q)  # the question
    print('Tools:', r['calls'], '| mode:', r['mode'])  # which tools ran
    print('A:', r['answer'])  # the answer
    print()  # blank line between answers

Q: Give me the summary for ACC2003
Tools: ['account_summary'] | mode: offline
A: ACC2003 (MUR): 28 transactions, 27 successful and 1 failed. Credits 73,587.31, debits 52,018.72, net 21,568.59. Largest debit 9,990.00. Period 2026-07-14 to 2026-09-29.

Q: Show large debits over 10000
Tools: ['flag_high_value_debits'] | mode: offline
A: 3 successful debit(s) at or above 10,000 MUR, largest first:
- TXN0209 (ACC2001, 2026-09-26 09:40): 22,008.33 MUR
- TXN0090 (ACC2002, 2026-08-14 11:02): 12,500.00 MUR
- TXN0150 (ACC2004, 2026-09-03 15:40): 10,800.00 MUR

Q: Failed transactions on ACC2002
Tools: ['search_transactions'] | mode: offline
A: 2 match(es), showing 2:
- TXN0165 2026-09-08 DEBIT 5,000.00 MUR (FAILED)
- TXN0039 2026-07-20 DEBIT 499.38 MUR (FAILED)

Q: What accounts do you have?
Tools: ['list_accounts'] | mode: offline
A: Accounts: ACC2001 (MUR, 35 transactions); ACC2002 (MUR, 44 transactions); ACC2003 (MUR, 28 transactions); ACC2004 (MUR, 38 transactions); ACC2005 (MUR, 34 transacti

## 4. Guardrail
Every number in an answer must appear in a tool result. Here is a check on a correct answer and an invented one.

In [7]:
evidence = ['{"account_id": "ACC2003", "net_flow": 21568.59, "successful": 27}']  # what a tool actually returned, including the account ID
good = 'ACC2003 has 27 successful transactions and a net flow of 21,568.59.'  # every number is in the evidence
bad = 'ACC2003 has 27 successful transactions and a net flow of 25,000.00.'  # 25,000.00 was never returned
print('Good answer passes:', bank_assistant.passes_guardrail(good, evidence))  # expect True
print('Invented number passes:', bank_assistant.passes_guardrail(bad, evidence))  # expect False

Good answer passes: True
Invented number passes: False


## 5. Privacy check
The raw customer IDs are read here only to test the server. No tool output may contain them.

In [8]:
raw_ids = set(pd.read_csv(DATA, dtype=str)['customer_id'].dropna().str.strip())  # raw customer IDs from the source file
outputs = []  # every tool output we can collect
for name, args in [('list_accounts', {}), ('search_transactions', {'limit': 50}), ('flag_high_value_debits', {'threshold_mur': 0}), ('account_summary', {'account_id': 'ACC2001'})]:  # call each tool
    outputs.append(str(await assistant.call(name, args)))  # store the text of its output
leaks = [i for i in raw_ids if any(i in o for o in outputs)]  # any raw ID found in an output
print('Raw customer IDs in source:', len(raw_ids))  # how many there are
print('Raw customer IDs leaked:', len(leaks))  # must be zero
assert not leaks, f'Leak detected: {leaks[:3]}'  # stop the notebook if any leak exists

Raw customer IDs in source: 6
Raw customer IDs leaked: 0


## 6. Claude mode without an API key
To test the tool-use loop with no key, a scripted fake model stands in for Claude. It first asks for a tool, then answers using the result. The guardrail runs on the final answer.

In [9]:
from types import SimpleNamespace as NS  # tiny objects that look like API response blocks

class FakeMessages:  # mimics client.messages.create
    def __init__(self):  # script: first a tool request, then a final answer
        self.turn = 0  # which scripted reply we are on
    def create(self, **kwargs):  # called by the assistant on each round
        self.turn += 1  # move to the next scripted reply
        if self.turn == 1:  # round 1: ask for a tool
            block = NS(type='tool_use', name='account_summary', input={'account_id': 'ACC2003'}, id='toolu_1')
            return NS(stop_reason='tool_use', content=[block])
        final = 'ACC2003 has 27 successful transactions and a net flow of 21,568.59 MUR.'  # round 2: answer
        return NS(stop_reason='end_turn', content=[NS(type='text', text=final)])

class FakeClient:  # stands in for anthropic.Anthropic()
    def __init__(self):  # create the fake messages object
        self.messages = FakeMessages()

test_assistant = await bank_assistant.BankAssistant(SERVER, DATA).start()  # a separate connection for the test
test_assistant.mode = 'claude'  # force claude mode
test_assistant.client = FakeClient()  # use the scripted model
claude_result = await test_assistant.ask('How is ACC2003 doing?')  # run the loop
print('Tools called:', claude_result['calls'])  # expect ['account_summary']
print('Answer:', claude_result['answer'])  # expect the final text
await test_assistant.stop()  # close the test connection

Tools called: ['account_summary']
Answer: ACC2003 has 27 successful transactions and a net flow of 21,568.59 MUR.


## 7. Tests
These checks must pass before the project is considered working.

In [10]:
failures = []  # collect failed checks instead of stopping at the first one
def check(name, ok):  # record one result
    print(('PASS ' if ok else 'FAIL ') + name)  # print the outcome
    if not ok:  # keep track of failures
        failures.append(name)

names = {t.name for t in assistant.tools}  # tool names the server exposes
check('server exposes four tools', names == {'list_accounts', 'account_summary', 'search_transactions', 'flag_high_value_debits'})
check('summary net equals credits minus debits', abs(summary['net_flow'] - (summary['credits'] - summary['debits'])) < 0.01)
check('large debits are all at or above the threshold', all(r['amount'] >= 10000 for r in large['rows']))
check('large debits are sorted largest first', [r['amount'] for r in large['rows']] == sorted([r['amount'] for r in large['rows']], reverse=True))
check('search respects the limit', (await assistant.call('search_transactions', {'limit': 3}))['returned'] == 3)
check('unknown account returns an error', 'error' in await assistant.call('account_summary', {'account_id': 'ACC9999'}))
check('router sends totals questions to account_summary', bank_assistant.route_offline('summary for acc2003', [])[0] == 'account_summary')
check('router sends large-debit questions to flag_high_value_debits', bank_assistant.route_offline('large debits', [])[0] == 'flag_high_value_debits')
check('guardrail accepts a correct answer', bank_assistant.passes_guardrail(good, evidence))
check('guardrail rejects an invented number', not bank_assistant.passes_guardrail(bad, evidence))
check('no raw customer IDs leak', len(leaks) == 0)
check('Claude loop calls the tool and passes the guardrail', claude_result['calls'] == ['account_summary'] and 'Guardrail' not in claude_result['answer'])
print()
print('All tests passed' if not failures else f'Failed: {failures}')  # overall result
assert not failures, failures  # stop the notebook on any failure

PASS server exposes four tools
PASS summary net equals credits minus debits
PASS large debits are all at or above the threshold
PASS large debits are sorted largest first
PASS search respects the limit
PASS unknown account returns an error
PASS router sends totals questions to account_summary
PASS router sends large-debit questions to flag_high_value_debits
PASS guardrail accepts a correct answer
PASS guardrail rejects an invented number
PASS no raw customer IDs leak
PASS Claude loop calls the tool and passes the guardrail

All tests passed


## 8. Optional: live Claude
Runs only if `ANTHROPIC_API_KEY` is set. Without a key this cell prints a message and does nothing.

In [11]:
if assistant.mode == 'claude':  # the key is set, so the live model is used
    live = await assistant.ask('Which account had the largest debit, and how much was it?')  # a question that needs tools
    print('Tools:', live['calls'])  # the tools Claude chose
    print('Answer:', live['answer'])  # the answer, checked by the guardrail
else:  # no key
    print('Skipped: set ANTHROPIC_API_KEY to test the live model.')  # explain how to enable it

Skipped: set ANTHROPIC_API_KEY to test the live model.


## 9. Shut down

In [12]:
await assistant.stop()  # close the MCP server connection
print('Server stopped.')  # confirm

Server stopped.
